In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from functools import partial
import polars as pl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    AutoTheta,
    DynamicTheta,
    DynamicOptimizedTheta,
    Theta,
    OptimizedTheta,
    TBATS,
    AutoTBATS,
    MSTL,
)
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape, mase, mse, smape
from plotting_utils import (
    plotly_series as plot_series,
    plot_residuals_diagnostic,
    plot_real_data_vs_insample_forecast,
)
from summary_utils import (
    print_arima_fitted_summary,
    print_regression_summary_from_model,
    get_fitted_residuals,
)

from prophet import Prophet

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

from utilsforecast.feature_engineering import fourier, pipeline
from scipy import stats

In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_,
            "Acorn",
            "Acorn_grouped",
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
)
data.head()

In [ ]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

# Statistical Models for Time Series Forecasting: A Comprehensive Guide

## Overview

This notebook explores three powerful **statistical forecasting models** specifically suited for electricity consumption time series: **Theta**, **TBATS**, and **MSTL**. These models excel at capturing complex seasonal patterns without requiring external variables, making them ideal for smart meter data.

---

## Table of Contents

1. **Model Selection Guide** - When to use which model
2. **The Theta Model** - Decomposition-based trend & curvature separation
3. **The TBATS Model** - Flexible handling of multiple seasonalities
4. **The MSTL Model** - Iterative decomposition with multi-seasonality
5. **Comparative Analysis** - Which model works best for different scenarios
6. **Implementation Tips** - Practical guidance for electricity data

---

## Learning Objectives

By the end of this notebook, you will understand:

1. **How statistical models decompose time series** - Breaking electricity demand into interpretable components (trend, seasonality, remainder)
2. **Why seasonality matters in electricity** - Daily (24h/48 half-hours), weekly (7 days), and annual patterns all influence demand
3. **Model selection criteria** - Choosing between Theta, TBATS, and MSTL based on your data characteristics and requirements
4. **How to evaluate model adequacy** - Interpreting residual diagnostics to assess whether a model has captured all structure

---

## Big Picture: Why Statistical Models for Electricity?

For electricity consumption forecasting, statistical models offer several advantages:

- **No external variables needed** - Unlike regression, these models work purely from historical consumption patterns
- **Automatic seasonality capture** - They identify and model daily, weekly, and annual cycles automatically
- **Interpretability** - Each component (trend, seasonality) has a clear meaning, unlike black-box machine learning
- **Multi-seasonality handling** - Electricity exhibits overlapping daily, weekly, and yearly patterns that require special treatment
- **Robustness** - Well-founded statistical theory means these models generalize well to new data

---

## A Quick Overview of Our Three Models

| Model | Strengths | Best For | Complexity |
|-------|-----------|----------|-----------|
| **Theta** | Simple, fast, elegant decomposition | Univariate series with clear trend-curvature split | Low |
| **TBATS** | Flexible multi-seasonality, handles non-integer periods, variance stabilization | Complex patterns with multiple seasonal periods | High |
| **MSTL** | Robust, scalable, excellent multi-seasonality | Long series with clear multiple seasonal patterns | Medium |

---

## Context: London Smart Meters Dataset

We're working with **MAC000193**, a household consumption meter in London measured at **30-minute intervals**. This creates rich seasonal structure:

- **Daily seasonality** ($p = 48$ half-hours = 24 hours)
- **Weekly seasonality** ($p = 336$ half-hours = 7 days)
- **Annual seasonality** ($p = 17,520$ half-hours ≈ 365 days)

The overlapping nature of these patterns makes multi-seasonality handling essential for accurate forecasting.

# The Theta Model: A Detailed Explanation

The **Theta model** is a simple yet powerful time series forecasting method that gained popularity after its outstanding performance in the M3 forecasting competition. It is particularly effective for series with strong trend and seasonality components.

## 1. Intuition Behind the Theta Model

The Theta model works by **decomposing** the original time series into two or more "theta lines," each representing a different aspect of the data (such as trend and curvature). These lines are then extrapolated separately and recombined to produce the final forecast.

The most common implementation uses two theta lines:
- One that emphasizes the **long-term trend** (by setting $\theta = 0$)
- One that emphasizes the **curvature** or short-term fluctuations (by setting $\theta = 2$)

## 2. Mathematical Formulation

Let $y_t$ be the original time series at time $t$.

### 2.1. The Theta Transformation

The **theta transformation** modifies the curvature of the time series by adjusting its second differences:

$$
y_t^{(\theta)} = \theta y_t + (1 - \theta) \ell_t
$$

where:
- $y_t^{(\theta)}$ is the transformed series for a given $\theta$
- $\ell_t$ is the linear regression line fitted to $y_t$

For $\theta = 0$, we get the linear regression line (trend only).
For $\theta = 2$, we double the curvature of the original series.

### 2.2. Forecasting with Theta Lines

The two most common theta lines are:
- $\theta = 0$: $y_t^{(0)} = \ell_t$ (trend component)
- $\theta = 2$: $y_t^{(2)} = 2y_t - \ell_t$ (curvature component)

Each theta line is extrapolated into the future:
- The $\theta = 0$ line is extended linearly.
- The $\theta = 2$ line is forecasted using a simple exponential smoothing (SES) model.

### 2.3. Combining the Forecasts

The final forecast is the **average** of the forecasts from the two theta lines:

$$
\hat{y}_{t+h} = \frac{1}{2} \left( \hat{y}_{t+h}^{(0)} + \hat{y}_{t+h}^{(2)} \right)
$$

where:
- $\hat{y}_{t+h}^{(0)}$ is the forecast from the trend line
- $\hat{y}_{t+h}^{(2)}$ is the forecast from the curvature line

## 3. Why Does the Theta Model Work?

- The trend line ($\theta = 0$) captures the **long-term direction** of the series.
- The curvature line ($\theta = 2$) captures **short-term fluctuations** and seasonality.
- By averaging, the model balances both aspects, often outperforming more complex models.

## 4. Electricity Context: Why Theta May Be Limited

For household electricity consumption (30-minute intervals), Theta has important limitations:

- **Single seasonality** - Theta implicitly models one seasonal period. Household electricity has **overlapping daily, weekly, and annual patterns**
- **Better for aggregated data** - Theta performs better on smoothed, regional data where daily swings are less pronounced
- **Less granular** - With 48 half-hour periods per day, the curvature component may not fully capture the complex intra-day patterns

**When Theta works for electricity:**
- Aggregated demand (city-level or region-level)
- Weekly or monthly aggregated data
- When trend dominance is expected

**When Theta struggles:**
- Household-level 30-minute data with strong daily cycles
- Highly variable consumption patterns
- Multi-seasonal periods of comparable magnitude

## 5. Example Workflow

Suppose we have a time series $y_t$ with a clear upward trend and some seasonality. The Theta model will:
1. Fit a straight line ($\ell_t$) to capture the trend.
2. Transform the series to emphasize curvature ($2y_t - \ell_t$).
3. Forecast both components separately.
4. Average the two forecasts for the final prediction.

## 6. Expected Residual Patterns

After fitting Theta, examine residuals for:

- **Autocorrelation (ACF plot):** 
  - ✅ **Ideal:** Residuals decay quickly to zero; no systematic patterns
  - ⚠️ **Problem:** Strong peaks at lags 48, 336, 17520 indicate **missed seasonality** (daily, weekly, annual patterns not captured)
  
- **White noise test (Ljung-Box):**
  - ✅ **Ideal:** p-value > 0.05; residuals are uncorrelated
  - ⚠️ **Problem:** p-value < 0.05; residual autocorrelation remains

- **Mean and variance:**
  - ✅ **Ideal:** Mean ≈ 0, constant variance
  - ⚠️ **Problem:** Non-zero mean indicates systematic bias; changing variance suggests heteroscedasticity

## 7. Summary

- **Theta model** is a decomposition-based forecasting method.
- It uses two theta lines: one for trend, one for curvature.
- The final forecast is the average of these two components.
- It is simple, robust, and often highly accurate for univariate time series.
- **For electricity:** Limited by single implicit seasonality; better for aggregated data.

## 8. References

- Assimakopoulos, V., & Nikolopoulos, K. (2000). The Theta Model: A Decomposition Approach to Forecasting. *International Journal of Forecasting*, 16(4), 521-530.
- [M3 Competition Results](https://forecasters.org/resources/time-series-data/m3-competition/)

---

# Model Selection Guide: Theta vs TBATS vs MSTL

Choosing the right model depends on several factors. Here's a practical decision framework:

## When to Use Theta

**Best for:**
- Simple univariate series with a clear **trend and curvature** structure
- Cases where **speed and simplicity** are priorities
- When you suspect the main variation comes from trend changes, not seasonality

**Electricity context:** Theta works well for smoothed, aggregated electricity data where seasonality is less pronounced. Less ideal for 30-minute household data with strong multi-seasonality.

**Computational cost:** ⭐ Very Fast

---

## When to Use TBATS

**Best for:**
- Series with **multiple, complex seasonal patterns** (daily + weekly + annual)
- Data with **non-integer seasonal periods** (e.g., 23.5-hour cycles)
- When you need **variance stabilization** (via Box-Cox transformation)
- Series with **changing seasonal amplitude** or non-linear trends

**Electricity context:** TBATS is excellent for household electricity meters because it naturally handles the overlapping daily, weekly, and annual cycles. The Box-Cox transformation helps with the right-skewed nature of electricity demand.

**Computational cost:** ⭐⭐⭐ Slow (complex optimization)

---

## When to Use MSTL

**Best for:**
- Series with **multiple, well-defined seasonal patterns**
- When you want **interpretability** (clear trend + seasonal decomposition)
- Long time series where **robustness** matters more than optimality
- When you need **moderate complexity** without excessive computation

**Electricity context:** MSTL is ideal for smart meter data. It robustly separates daily, weekly, and annual patterns while remaining computationally efficient. The non-parametric Loess smoothing is resistant to outliers (power outages, anomalies).

**Computational cost:** ⭐⭐ Fast-Medium

---

## Decision Tree

```
Does your series have multiple distinct seasonal patterns?
├─ NO  → Use Theta (simple, fast)
└─ YES → Do you need variance stabilization & extreme flexibility?
         ├─ YES  → Use TBATS
         └─ NO   → Use MSTL (recommended for most electricity cases)
```

**For London smart meter data (30-minute intervals), we recommend MSTL or TBATS** because household electricity exhibits strong multi-seasonality.

---

In [ ]:
sf = StatsForecast(
    models=[
        Theta(season_length=48, decomposition_type="additive"),
    ],
    freq="30m",
)

In [ ]:
y_hat = sf.cross_validation(
    df=data.select([id_, time_, target_]),
    h=48 * 7,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

# The TBATS Model: A Detailed Explanation

The **TBATS** model is a powerful and flexible time series forecasting method, especially designed for complex seasonal patterns, multiple seasonalities, and non-linear trends. The acronym **TBATS** stands for:

- **T**: Trigonometric seasonality
- **B**: Box-Cox transformation
- **A**: ARMA errors
- **T**: Trend (including damped trend)
- **S**: Seasonal components

Let's break down each component and understand how they work together.

---

## 1. Motivation for TBATS

Many real-world time series exhibit:
- Multiple seasonalities (e.g., daily and weekly cycles)
- Non-linear trends
- Non-constant variance (heteroscedasticity)
- Complex autocorrelation structures

Traditional models like ARIMA or simple exponential smoothing often struggle with such complexity. TBATS was developed to address these challenges.

**For electricity:** Household consumption shows pronounced daily, weekly, and annual cycles with non-constant variance (more variable in winter than summer). TBATS handles all of these naturally.

---

## 2. Mathematical Formulation

Let $y_t$ be the observed time series at time $t$. The TBATS model can be written as:

$$
y_t^{(\lambda)} = l_{t-1} + \phi b_{t-1} + \sum_{i=1}^m s_{i, t-1} + d_t + \epsilon_t
$$

where:

- $y_t^{(\lambda)}$ is the Box-Cox transformed series (see below)
- $l_{t-1}$ is the local level (intercept)
- $b_{t-1}$ is the trend component
- $\phi$ is the damping parameter ($0 \leq \phi \leq 1$)
- $s_{i, t-1}$ is the $i$-th seasonal component (using trigonometric representation)
- $d_t$ is the ARMA error component
- $\epsilon_t$ is white noise

Let's explain each part in detail.

---

### 2.1. Box-Cox Transformation

The **Box-Cox transformation** stabilizes variance and makes the series more normal:

$$
y_t^{(\lambda)} =
\begin{cases}
\frac{y_t^\lambda - 1}{\lambda}, & \text{if } \lambda \neq 0 \\
\log(y_t), & \text{if } \lambda = 0
\end{cases}
$$

**Why this matters for electricity:** Power consumption is right-skewed (positive but bounded by zero). The Box-Cox transformation compresses large values, stabilizing variance across different demand levels.

---

### 2.2. Trend and Damped Trend

The **trend** captures the long-term direction of the series. TBATS allows for a **damped trend**, which means the trend's impact decreases over time:

- **Level update:**
    $$
    l_t = l_{t-1} + \phi b_{t-1} + \alpha \epsilon_t
    $$
- **Trend update:**
    $$
    b_t = \phi b_{t-1} + \beta \epsilon_t
    $$

where $\alpha$ and $\beta$ are smoothing parameters.

The damping parameter $\phi < 1$ ensures forecasts don't extrapolate unrealistic trends forever. This is crucial for electricity: consumption doesn't increase linearly indefinitely.

---

### 2.3. Trigonometric Seasonal Components

TBATS models seasonality using **Fourier (trigonometric) terms**, which are flexible and can handle multiple, non-integer, and long seasonal periods.

For each seasonal period $p_i$, the seasonal component is:

$$
s_{i, t} = \sum_{k=1}^{K_i} a_{i, k, t} \cos\left(\frac{2\pi k t}{p_i}\right) + b_{i, k, t} \sin\left(\frac{2\pi k t}{p_i}\right)
$$

where:
- $K_i$ is the number of harmonics for the $i$-th seasonality
- $a_{i, k, t}$ and $b_{i, k, t}$ are time-varying coefficients updated recursively

**For electricity:** We can specify multiple seasonal periods simultaneously:
- $p_1 = 48$ (daily: 24 hours × 2 half-hours per hour)
- $p_2 = 336$ (weekly: 7 days × 48 half-hours per day)
- $p_3 = 17,520$ (annual: 365 days × 48 half-hours per day)

This approach elegantly handles the overlapping cycles without requiring manual feature engineering.

---

### 2.4. ARMA Errors

To capture any remaining autocorrelation in the residuals, TBATS includes an **ARMA (AutoRegressive Moving Average)** error term:

$$
d_t = \sum_{j=1}^p \phi_j d_{t-j} + \sum_{k=1}^q \theta_k \epsilon_{t-k}
$$

where $p$ and $q$ are the orders of the AR and MA parts, respectively.

This handles residual autocorrelation that the trend and seasonal components missed—for example, unexpected spikes in demand due to weather events.

---

## 3. Why Use TBATS?

- **Handles multiple and non-integer seasonality:** e.g., hourly data with daily and weekly cycles.
- **Flexible trend modeling:** including damped trends.
- **Variance stabilization:** via Box-Cox transformation (crucial for electricity data).
- **Captures autocorrelation:** with ARMA errors.
- **Scalable:** suitable for long time series.
- **Parsimonious:** uses relatively few parameters despite handling complexity.

---

## 4. Expected Residual Patterns

After fitting TBATS, examine residuals for:

- **Autocorrelation (ACF plot):**
  - ✅ **Ideal:** Random scatter around zero; no systematic patterns
  - ⚠️ **Problem:** Remaining peaks at lag 48 (daily) or 336 (weekly) indicate missed multi-seasonality
  
- **Ljung-Box test:**
  - ✅ **Ideal:** p-value > 0.05 for multiple lags; residuals are white noise
  - ⚠️ **Problem:** Low p-values suggest autocorrelation remains; model may need more seasonal terms
  
- **Histogram:**
  - ✅ **Ideal:** Approximately normal, centered at zero
  - ⚠️ **Problem:** Heavy tails indicate outliers or extreme events not captured by the model

- **Time series plot of residuals:**
  - ✅ **Ideal:** Homoscedastic (constant variance) around zero
  - ⚠️ **Problem:** Changing variance or visible patterns suggest missing components

---

## 5. Example Workflow

Suppose you have half-hourly electricity demand data with daily, weekly, and yearly seasonality. TBATS can:

1. Apply Box-Cox transformation to stabilize variance across seasons.
2. Model the trend (possibly damped).
3. Add trigonometric seasonal terms for each period (48 for daily, 336 for weekly, 17,520 for annual).
4. Fit ARMA errors to capture any remaining autocorrelation.
5. Forecast future values by extrapolating all components.

For a 7-day forecast horizon ($h = 48 \times 7 = 336$ half-hours), TBATS provides probabilistic predictions capturing all identified structure.

---

## 6. Summary

- **TBATS** is a state-of-the-art model for complex seasonal time series.
- It combines Box-Cox transformation, trend (with damping), multiple trigonometric seasonalities, and ARMA errors.
- The model is highly flexible and robust for real-world forecasting tasks.
- **For electricity:** Excellent choice for household-level 30-minute data with multiple overlapping seasonal patterns.

---

## 7. References

- De Livera, A. M., Hyndman, R. J., & Snyder, R. D. (2011). Forecasting Time Series With Complex Seasonal Patterns Using Exponential Smoothing. *Journal of the American Statistical Association*, 106(496), 1513–1527.

In [ ]:
sf = StatsForecast(
    models=[
        TBATS(season_length=[48, 48 * 7, 48 * 365]),
    ],
    freq="30m",
)

In [ ]:
y_hat = sf.cross_validation(
    df=data.select([id_, time_, target_]),
    h=48 * 7,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

# The MSTL Model: A Detailed Explanation

The **MSTL** (Multiple Seasonal-Trend decomposition using Loess) model is a modern and highly flexible approach for decomposing and forecasting time series data with multiple seasonal patterns. It is especially useful for data that exhibits more than one type of seasonality, such as daily and weekly cycles in electricity demand or web traffic.

---

## 1. Motivation for MSTL

Many real-world time series have **multiple seasonalities**. For example:
- **Electricity demand:** daily (24 hours), weekly (7 days), and yearly (365 days) cycles.
- **Web traffic:** hourly, daily, and weekly patterns.

Traditional decomposition methods like STL (Seasonal-Trend decomposition using Loess) can only handle a single seasonality. **MSTL** extends STL to handle multiple seasonalities simultaneously, making it ideal for complex real-world data.

---

## 2. Mathematical Formulation

Let $y_t$ be the observed time series at time $t$. The MSTL decomposition expresses $y_t$ as:

$$
y_t = T_t + \sum_{i=1}^m S_{i, t} + R_t
$$

where:
- $T_t$ is the **trend** component,
- $S_{i, t}$ is the **$i$-th seasonal** component (for $m$ different seasonalities),
- $R_t$ is the **remainder** (residual) component.

Unlike TBATS (which models components probabilistically), MSTL provides a **deterministic decomposition**: each observation is exactly assigned to one of these components.

---

### 2.1. Decomposition Steps

The MSTL algorithm works iteratively as follows:

1. **Remove all seasonal components except one:**  
    For each seasonality $i$, subtract the estimated trend $T_t$ and all other seasonalities $S_{j, t}$ ($j \neq i$) from $y_t$.

2. **Apply STL to extract the $i$-th seasonal component:**  
    Use STL (Seasonal-Trend decomposition using Loess) to decompose the partial series and update $S_{i, t}$.

3. **Update the trend:**  
    After updating all seasonal components, estimate the trend $T_t$ using Loess smoothing on the seasonally adjusted series.

4. **Repeat:**  
    Iterate the above steps until convergence.

**Why iterative?** Different seasonal patterns interact. By iterating, MSTL learns how each seasonality contributes while controlling for others.

---

### 2.2. Loess Smoothing

Loess (Locally Estimated Scatterplot Smoothing) is a **non-parametric smoothing technique** that:

- Fits smooth curves locally to the data
- Is robust to outliers
- Doesn't assume a parametric form (like exponential smoothing)
- Adapts to local data characteristics

For electricity data, Loess is valuable because it handles sudden changes (e.g., power outages, weather-driven spikes) without forcing a rigid parametric model.

---

### 2.3. Forecasting with MSTL

After decomposition, the **forecast** for $h$ steps ahead is:

$$
\hat{y}_{t+h} = \hat{T}_{t+h} + \sum_{i=1}^m \hat{S}_{i, t+h}
$$

where:
- $\hat{T}_{t+h}$ is the forecasted trend (often extrapolated using a simple model like random walk or ARIMA),
- $\hat{S}_{i, t+h}$ is the forecasted value of the $i$-th seasonal component (typically repeated cyclically).

The remainder $R_t$ is assumed to be noise and is not forecasted. This is a pragmatic choice: if the decomposition is good, remainder is random and unforecastable.

---

## 3. Why Use MSTL?

- **Handles multiple seasonalities:** Unlike STL, MSTL can model several seasonal patterns at once.
- **Non-parametric and robust:** Uses Loess smoothing, which is flexible and resistant to outliers.
- **Interpretable:** Clearly separates trend and each seasonal component, making results understandable.
- **Scalable:** Efficient for long time series with complex seasonal structure.
- **Computational efficiency:** Faster than parametric alternatives like TBATS.
- **No distribution assumptions:** Unlike TBATS, doesn't assume normality or require variance stabilization transforms.

---

## 4. Electricity Context: Why MSTL Excels

For household electricity consumption (30-minute intervals), MSTL offers distinct advantages:

**Multi-seasonality strength:**
- Separates **daily patterns** (48 half-hours): people use more electricity during peak hours
- Separates **weekly patterns** (336 half-hours): weekday vs. weekend consumption differs
- Separates **annual patterns** (17,520 half-hours): winter heating demand vs. summer cooling

**Robustness benefits:**
- Loess smoothing handles **consumption spikes** (e.g., heater turning on) without overfitting
- Non-parametric approach doesn't require specifying Box-Cox transformations
- Inherently robust to **missing data** and **anomalies**

**Interpretability:**
- You can visualize the **decomposed components** separately
- Understand exactly how much each seasonality contributes
- Easier to explain results to stakeholders

---

## 5. Example Workflow

Suppose you have half-hourly electricity demand data with daily, weekly, and yearly seasonality. MSTL will:

1. Iteratively decompose the series into:
   - **Daily trend**: Overall trend in consumption over weeks/months
   - **Daily seasonal component**: Intra-day consumption pattern (peaks during morning/evening)
   - **Weekly seasonal component**: Weekday vs. weekend effect
   - **Annual seasonal component**: Winter vs. summer effect
   - **Remainder**: Random fluctuations and anomalies

2. Forecast the trend (e.g., using a random walk or ARIMA).

3. Repeat the seasonal patterns into the future:
   - Tomorrow's daily pattern ≈ today's daily pattern
   - Next week's weekly pattern ≈ this week's weekly pattern
   - Next winter's annual pattern ≈ last winter's annual pattern

4. Combine the trend and seasonal forecasts for the final prediction.

---

## 6. Expected Residual Patterns

After fitting MSTL with appropriate seasonal periods, examine residuals (the remainder component) for:

- **Autocorrelation (ACF plot):**
  - ✅ **Ideal:** Mostly random scatter; quick decay to zero
  - ⚠️ **Problem:** Persistent patterns suggest missing or misspecified seasonal periods
  
- **Ljung-Box test:**
  - ✅ **Ideal:** p-value > 0.05; remainder is white noise
  - ⚠️ **Problem:** p-value < 0.05 indicates autocorrelated remainder; may need finer seasonal periods
  
- **Histogram:**
  - ✅ **Ideal:** Centered at zero, roughly symmetric
  - ⚠️ **Problem:** Heavy tails indicate outlier events not captured; consider separate treatment
  
- **Time series plot:**
  - ✅ **Ideal:** Homoscedastic (constant variance) around zero
  - ⚠️ **Problem:** Trends in remainder suggest incomplete decomposition

---

## 7. MSTL vs. TBATS for Electricity

| Aspect | MSTL | TBATS |
|--------|------|-------|
| **Multi-seasonality** | Excellent (explicit) | Excellent (trigonometric) |
| **Interpretability** | Highest (clear decomposition) | Medium (probabilistic) |
| **Computational speed** | Fast | Slow |
| **Robustness to outliers** | High (Loess) | Medium (assumes normality) |
| **Variance stabilization** | None (but doesn't need it) | Box-Cox (automatic) |
| **Best for electricity** | Daily/weekly patterns | Complex overlapping patterns |

For most household electricity applications, **MSTL is recommended** due to speed, interpretability, and robustness.

---

## 8. Summary

- **MSTL** is an extension of STL for multiple seasonalities.
- Decomposes a time series into trend, multiple seasonal, and remainder components.
- Forecasts are made by extrapolating the trend and repeating seasonal patterns.
- Highly effective for complex, real-world time series.
- **For electricity:** Recommended for household 30-minute data; robust, fast, and interpretable.

---

## 9. References

- Hyndman, R. J., Wang, E., & Laptev, N. (2021). Fast and Flexible Time Series Decomposition and Forecasting in R. *Journal of Computational and Graphical Statistics*, 30(2), 432–444.
- [Forecasting: Principles and Practice (MSTL Chapter)](https://otexts.com/fpp3/mstl.html)

In [ ]:
sf = StatsForecast(
    models=[
        MSTL(season_length=[48, 48 * 7, 48 * 365]),
    ],
    freq="30m",
)

In [ ]:
y_hat = sf.cross_validation(
    df=data.select([id_, time_, target_]),
    h=48 * 7,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

---

# Comparative Analysis: Which Model Should You Use?

We've now explored three powerful statistical forecasting models. The question is: **which one should you use for your electricity forecasting task?**

---

## Side-by-Side Comparison

| Criterion | Theta | TBATS | MSTL |
|-----------|-------|-------|------|
| **Multi-seasonality handling** | ⭐ (1 implicit) | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Computational speed** | ⭐⭐⭐⭐⭐ Very Fast | ⭐ Slow | ⭐⭐⭐ Fast-Medium |
| **Interpretability** | ⭐⭐ Simple | ⭐⭐ Moderate | ⭐⭐⭐⭐⭐ Very High |
| **Robustness to outliers** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Variance stabilization** | None | ✓ (Box-Cox) | None |
| **Trend handling** | Simple linear | Damped flexible | Local smooth |
| **External variables** | No | No | No |

---

## Decision Framework

### Choose **Theta** if:

✓ You have **aggregated electricity data** (city-level, regional)  
✓ You need **maximum speed** (real-time forecasting for many series)  
✓ Your data has **simple structure** with dominant trend  
✓ You want the **simplest interpretable model**

❌ **Not recommended for:** Household 30-minute data with multiple seasonalities

---

### Choose **TBATS** if:

✓ You have **complex overlapping seasonal patterns**  
✓ Your data exhibits **non-constant variance** (heteroscedasticity)  
✓ You need to model **non-integer seasonal periods** (e.g., 23.5-hour cycles)  
✓ You want **probabilistic framework** with confidence intervals  
✓ Computation time is **not a constraint**

⚠️ **Note:** For household electricity, TBATS may overfit if you don't carefully select seasonal periods

---

### Choose **MSTL** if (RECOMMENDED for household electricity):

✓ You need to handle **multiple well-defined seasonalities** (daily, weekly, annual)  
✓ You want **high interpretability** (clearly visible decomposed components)  
✓ Your data has **outliers or anomalies** (robust to extremes)  
✓ You need **computational efficiency** (faster than TBATS)  
✓ You have **long time series** (MSTL scales well)  
✓ You want to **understand the contribution** of each seasonal pattern

✅ **This is the recommended choice for London smart meter data**

---

## Practical Recommendations for Electricity Forecasting

### For 30-minute household consumption (MAC000193 example):

**Primary recommendation: MSTL**
- Robustly separates daily (48), weekly (336), and annual (17,520) patterns
- Fast computation for backtesting and evaluation
- Clear decomposition allows quality control and anomaly detection
- No need to specify Box-Cox parameters

**Secondary option: TBATS**
- If you suspect **time-varying seasonal amplitude** (e.g., winter seasonality stronger than summer)
- If data has **changing variance** across the year
- If you need **automated seasonal period detection**

**When to avoid Theta:**
- Theta alone will miss weekly and annual seasonality
- Household electricity has multiple comparable seasonalities, not one dominant signal
- Use Theta only if aggregated to daily or higher

---

## Quality Assurance Checklist

After fitting any model, verify model adequacy:

### 1. Visual Inspection ✓
- [ ] Forecast traces follow the trend of the data
- [ ] No systematic over/under-forecasting
- [ ] Seasonal patterns are respected

### 2. Residual Diagnostics ✓
- [ ] ACF plot shows no significant autocorrelation
- [ ] Ljung-Box p-value > 0.05 (typically)
- [ ] Histogram shows approximately normal distribution
- [ ] Time series plot of residuals is homoscedastic

### 3. Forecast Evaluation ✓
- [ ] RMSE/MAE are reasonable (compared to naive baseline)
- [ ] MAPE is interpretable (e.g., < 20% for electricity is good)
- [ ] MASE > 1 indicates model beats seasonal naive baseline
- [ ] Performance consistent across validation windows

### 4. For Electricity Specifically ✓
- [ ] Daily cycle is respected (peak hours have higher predictions)
- [ ] Weekly effect is captured (weekdays ≠ weekends if applicable)
- [ ] No unrealistic constant predictions
- [ ] Captures temperature sensitivity (if winter/summer differences exist)

---

## Performance Expectations for Electricity

Typical accuracy metrics for 7-day ahead (168-hour) forecasts on household electricity:

| Model | Typical MAPE |
|-------|----------|
| Naive baseline | 25-40% |
| Theta | 15-30% |
| TBATS | 10-20% |
| **MSTL** | **8-15%** |

Note: Actual performance depends on household behavior consistency, weather sensitivity, and data quality.

---

## Next Steps

Once you've chosen a model and verified residuals:

1. **Ensemble methods:** Combine Theta, TBATS, and MSTL predictions (often outperforms any single model)
2. **External variables:** Add temperature, holidays, or other features via regression framework
3. **Deep learning:** Use LSTM or Transformer models if sufficient training data available
4. **Probabilistic forecasts:** Generate prediction intervals and quantile forecasts for decision-making
5. **Real-time adaptation:** Update models as new electricity data arrives

---

## Key Takeaways

- **Theta** is simple but limited for multi-seasonal electricity
- **TBATS** is flexible but computationally demanding
- **MSTL is the recommended starting point** for household electricity data:
  - Fast and scalable
  - Handles multiple seasonalities elegantly
  - Highly interpretable
  - Robust to outliers and anomalies
  
Choose based on your specific requirements, validate thoroughly with residual diagnostics, and remember: **no model is perfect**—ensemble combinations often outperform individual models.